# Phase 11_b — Attribution and Diagnostics

`11_b` is the mostly analysis-only notebook for responsible-step localization, failure-family analysis, and advisor-facing summaries built on top of the saved artifacts from `11_a`.


## Setup

In [ ]:
import os, sys, shutil

REPO_DIR = '/content/hallucination_detection'
if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)
if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b master https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes scipy scikit-learn')

from spectral_utils import (
    boot_auc,
    first_incorrect_step_index,
    categorize_failure_mode_v2,
)

import numpy as np
import pandas as pd
import pickle
print('analysis environment ready')


In [ ]:
BASE_DIR = '/content/drive/MyDrive/hallucination_detection/cache/phase11_agentic_v2'
RAW_DIR = os.path.join(BASE_DIR, 'raw')
FEAT_DIR = os.path.join(BASE_DIR, 'features')
RES_DIR_A = os.path.join(BASE_DIR, 'results_a')
RES_DIR_B = os.path.join(BASE_DIR, 'results_b')
PLOT_DIR_B = os.path.join(BASE_DIR, 'plots_b')
PHASE11A_PATH = os.path.join(RES_DIR_A, 'phase11a_results.pkl')
PHASE11B_PATH = os.path.join(RES_DIR_B, 'phase11b_results.pkl')

for d in (RES_DIR_B, PLOT_DIR_B):
    os.makedirs(d, exist_ok=True)

with open(PHASE11A_PATH, 'rb') as f:
    PHASE11A = pickle.load(f)

CONFIG = PHASE11A['config']
MODELS = CONFIG['models']
DATASETS = CONFIG['datasets']
AGGS = CONFIG['aggs']
CELL_KEYS = [tuple(k) if not isinstance(k, tuple) else k for k in PHASE11A['cell_keys']]
FEAT_KEYS = PHASE11A['feature_keys']
NADLER_RES = PHASE11A['NADLER_RES']
AUQ_RES = PHASE11A['AUQ_RES']
FUSION_RES = PHASE11A['FUSION_RES']
RHO_RES = PHASE11A['RHO_RES']
print('Loaded phase11a manifest:', PHASE11A_PATH)
print('Cells:', CELL_KEYS)


## Load per-cell artifacts

In [ ]:
def raw_path(model_key, dataset):
    return os.path.join(RAW_DIR, f'{model_key}__{dataset}.pkl')


def feat_path(model_key, dataset):
    return os.path.join(FEAT_DIR, f'{model_key}__{dataset}.pkl')


ALL_CELLS = {}
RAW_TRAJ = {}
for model_key, dataset in CELL_KEYS:
    with open(feat_path(model_key, dataset), 'rb') as f:
        ALL_CELLS[(model_key, dataset)] = pickle.load(f)
    with open(raw_path(model_key, dataset), 'rb') as f:
        RAW_TRAJ[(model_key, dataset)] = pickle.load(f)

print('Loaded feature cells:', len(ALL_CELLS))
print('Loaded raw trajectory cells:', len(RAW_TRAJ))


## Step localization (AgentHallu-style proxy)

In [ ]:
LSA_PATH = os.path.join(RES_DIR_B, 'lsa_res.pkl')
FORCE_RECOMPUTE_LSA = False

if not FORCE_RECOMPUTE_LSA and os.path.exists(LSA_PATH):
    with open(LSA_PATH, 'rb') as f:
        LSA_RES = pickle.load(f)
else:
    LSA_RES = {}

    z_params = {}
    vc_params = {}
    for model_key, dataset in CELL_KEYS:
        cell = ALL_CELLS[(model_key, dataset)]
        z_params[(model_key, dataset)] = {}
        all_steps = [step for traj_f in cell['step_features'] for step in traj_f if step is not None]
        for key in FEAT_KEYS:
            vals = np.asarray([step[key] for step in all_steps if key in step], dtype=float)
            if vals.size:
                z_params[(model_key, dataset)][key] = (float(np.nanmean(vals)), float(np.nanstd(vals)))
        vc_flat = [v for vc in cell['step_verb_conf'] for v in vc if v is not None and not np.isnan(v)]
        vc_params[(model_key, dataset)] = (
            float(np.mean(vc_flat)) if vc_flat else 0.0,
            float(np.std(vc_flat)) if vc_flat else 1.0,
        )

    def per_step_score(model_key, dataset, traj_i, step_k, detector):
        cell = ALL_CELLS[(model_key, dataset)]
        if step_k >= len(cell['step_features'][traj_i]):
            return float('nan')
        feat = cell['step_features'][traj_i][step_k]
        if detector == 'EPR':
            return -feat['epr'] if feat is not None and 'epr' in feat else float('nan')
        if detector == 'AUQ':
            v = cell['step_verb_conf'][traj_i][step_k]
            return v if v is not None else float('nan')
        if detector in ('SpectralNadler', 'Fused'):
            source = NADLER_RES if detector == 'SpectralNadler' else FUSION_RES
            res = source.get((model_key, dataset, 'min'), {})
            subset = res.get('subset') or []
            weights = res.get('weights') or []
            sign_map = NADLER_RES.get((model_key, dataset, 'min'), {}).get('sign_map', {})
            if not subset or feat is None:
                return float('nan')
            score = 0.0
            for j, key in enumerate(subset):
                if key == 'verb_conf':
                    v = cell['step_verb_conf'][traj_i][step_k]
                    if v is None:
                        return float('nan')
                    mu, sd = vc_params[(model_key, dataset)]
                    score += weights[j] * ((v - mu) / (sd + 1e-9))
                else:
                    if key not in feat:
                        return float('nan')
                    mu, sd = z_params[(model_key, dataset)].get(key, (0.0, 1.0))
                    z = (feat[key] - mu) / (sd + 1e-9)
                    score += weights[j] * sign_map.get(key, 1) * z
            return score
        return float('nan')

    for model_key, dataset in CELL_KEYS:
        cell = ALL_CELLS[(model_key, dataset)]
        raw = RAW_TRAJ[(model_key, dataset)]
        per_det = {'EPR': [], 'AUQ': [], 'SpectralNadler': [], 'Fused': []}
        per_family = {}
        n_evaluable = 0
        random_hits = 0.0
        for traj_i, (traj, traj_correct) in enumerate(zip(raw, cell['traj_labels'])):
            if traj_correct:
                continue
            gt = first_incorrect_step_index(traj)
            if gt is None:
                continue
            n_steps_valid = sum(1 for step in cell['step_features'][traj_i] if step is not None)
            if n_steps_valid < 2:
                continue
            n_evaluable += 1
            random_hits += 1.0 / n_steps_valid
            family = categorize_failure_mode_v2(traj)
            if family not in per_family:
                per_family[family] = {'n': 0, 'hits': {det: [] for det in per_det}}
            per_family[family]['n'] += 1
            for det in per_det:
                scores = [per_step_score(model_key, dataset, traj_i, k, det) for k in range(len(cell['step_features'][traj_i]))]
                valid = [(k, s) for k, s in enumerate(scores) if not np.isnan(s)]
                if len(valid) < 2:
                    continue
                pred = min(valid, key=lambda x: x[1])[0]
                hit = (pred == gt)
                per_det[det].append(hit)
                per_family[family]['hits'][det].append(hit)

        LSA_RES[(model_key, dataset)] = {
            det: {
                'phi_lsa': (float(np.mean(vals)) if vals else float('nan')),
                'n': len(vals),
            }
            for det, vals in per_det.items()
        }
        LSA_RES[(model_key, dataset)]['n_evaluable'] = n_evaluable
        LSA_RES[(model_key, dataset)]['random_baseline'] = (random_hits / n_evaluable) if n_evaluable else float('nan')
        LSA_RES[(model_key, dataset)]['per_family'] = {
            family: {
                'n': meta['n'],
                **{
                    det: (float(np.mean(meta['hits'][det])) if meta['hits'][det] else float('nan'))
                    for det in per_det
                },
            }
            for family, meta in per_family.items()
        }

    with open(LSA_PATH, 'wb') as f:
        pickle.dump(LSA_RES, f)

for key, res in sorted(LSA_RES.items()):
    print(key, 'random=', res['random_baseline'])
    for det in ('EPR', 'AUQ', 'SpectralNadler', 'Fused'):
        print(' ', det, res[det]['phi_lsa'], 'n=', res[det]['n'])


## Step-position and failure-family analysis

In [ ]:
STEP_PATH = os.path.join(RES_DIR_B, 'step_res.pkl')
FAILURE_PATH = os.path.join(RES_DIR_B, 'failure_res.pkl')
FORCE_RECOMPUTE_ANALYSIS = False

if not FORCE_RECOMPUTE_ANALYSIS and os.path.exists(STEP_PATH) and os.path.exists(FAILURE_PATH):
    with open(STEP_PATH, 'rb') as f:
        STEP_RES = pickle.load(f)
    with open(FAILURE_PATH, 'rb') as f:
        FAILURE_RES = pickle.load(f)
else:
    STEP_RES = {}
    FAILURE_RES = {}
    for model_key, dataset in CELL_KEYS:
        cell = ALL_CELLS[(model_key, dataset)]
        labels = cell['traj_labels']
        full_idx = [
            i for i, traj_f in enumerate(cell['step_features'])
            if len(traj_f) == CONFIG['max_steps'] and all(step is not None for step in traj_f)
        ]
        if len(full_idx) >= 30:
            full_idx = np.asarray(full_idx)
            labels_full = labels[full_idx]
            for feat in ('epr', 'verb_conf'):
                aucs = []
                for k in range(CONFIG['max_steps']):
                    if feat == 'verb_conf':
                        vals = np.asarray([cell['step_verb_conf'][i][k] for i in full_idx])
                    else:
                        vals = np.asarray([cell['step_features'][i][k][feat] for i in full_idx])
                    auc, _, _ = boot_auc(labels_full, vals)
                    if not np.isnan(auc) and auc < 0.5:
                        auc, _, _ = boot_auc(labels_full, -vals)
                    aucs.append(auc)
                STEP_RES[(model_key, dataset, feat)] = aucs

        families = cell.get('failure_modes_v2', [])
        counts = pd.Series(families).value_counts().to_dict()
        FAILURE_RES[(model_key, dataset)] = counts

    with open(STEP_PATH, 'wb') as f:
        pickle.dump(STEP_RES, f)
    with open(FAILURE_PATH, 'wb') as f:
        pickle.dump(FAILURE_RES, f)

print('STEP_RES cells:', len(STEP_RES))
print('FAILURE_RES cells:', len(FAILURE_RES))


In [ ]:
failure_rows = []
for (model_key, dataset), counts in sorted(FAILURE_RES.items()):
    total = sum(counts.values())
    row = {'model': model_key, 'dataset': dataset, 'total': total}
    for label in ('correct', 'planning', 'retrieval_query', 'retrieval_context', 'reasoning_or_synthesis', 'tool_or_format', 'no_finish'):
        row[label] = counts.get(label, 0)
    failure_rows.append(row)
df_failure = pd.DataFrame(failure_rows)
print(df_failure.to_string(index=False))

lsa_rows = []
for (model_key, dataset), res in sorted(LSA_RES.items()):
    row = {
        'model': model_key,
        'dataset': dataset,
        'n_evaluable': res['n_evaluable'],
        'random': res['random_baseline'],
    }
    for det in ('EPR', 'AUQ', 'SpectralNadler', 'Fused'):
        row[det] = res[det]['phi_lsa']
    lsa_rows.append(row)
df_lsa = pd.DataFrame(lsa_rows)
with pd.option_context('display.float_format', lambda x: f'{100*x:.1f}%' if isinstance(x, float) and not np.isnan(x) else '  -  '):
    print(df_lsa.to_string(index=False))


## Cross-dataset summary and advisor-facing outputs

In [ ]:
headline = pd.DataFrame(PHASE11A['headline_table'])
transfer_summary = []
for model_key in sorted(set(headline['model'])):
    sub = headline[(headline['model'] == model_key) & (headline['agg'] == 'Φ_min')]
    transfer_summary.append({
        'model': model_key,
        'best_spectral_nadler': sub['Spectral Nadler'].max(),
        'best_auq': sub['AUQ'].max(),
        'best_fused': sub['Spectral+AUQ'].max(),
    })
df_transfer = pd.DataFrame(transfer_summary)
with pd.option_context('display.float_format', lambda x: f'{100*x:.1f}%' if isinstance(x, float) and not np.isnan(x) else '  -  '):
    print(df_transfer.to_string(index=False))

advisor_rows = []
for (model_key, dataset), res in sorted(LSA_RES.items()):
    advisor_rows.append({
        'model': model_key,
        'dataset': dataset,
        'rho_epr_vs_conf': RHO_RES.get((model_key, dataset), {}).get('rho', float('nan')),
        'phi_lsa_fused': res['Fused']['phi_lsa'],
        'phi_lsa_auq': res['AUQ']['phi_lsa'],
        'best_fused_phi_min_auc': FUSION_RES.get((model_key, dataset, 'min'), {}).get('auc', float('nan')),
        'best_auq_phi_min_auc': AUQ_RES.get((model_key, dataset, 'min'), {}).get('auc', float('nan')),
    })
df_advisor = pd.DataFrame(advisor_rows)
with pd.option_context('display.float_format', lambda x: f'{100*x:.1f}%' if isinstance(x, float) and not np.isnan(x) else '  -  '):
    print(df_advisor.to_string(index=False))


In [ ]:
results_b = {
    'config': CONFIG,
    'LSA_RES': LSA_RES,
    'STEP_RES': STEP_RES,
    'FAILURE_RES': FAILURE_RES,
    'transfer_summary': df_transfer.to_dict(orient='records'),
    'advisor_table': df_advisor.to_dict(orient='records'),
}
with open(PHASE11B_PATH, 'wb') as f:
    pickle.dump(results_b, f)
print('Saved', PHASE11B_PATH)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
dets = ['EPR', 'AUQ', 'SpectralNadler', 'Fused']
plot_df = df_lsa.copy()
plot_df['cell'] = plot_df['model'] + '/' + plot_df['dataset']
vals = plot_df[dets].values
ax.imshow(vals, aspect='auto', cmap='magma', vmin=0.0, vmax=0.8)
ax.set_xticks(range(len(dets)))
ax.set_xticklabels(dets)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['cell'])
ax.set_title('Phase 11_b Φ_LSA heatmap')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR_B, 'phase11b_phi_lsa_heatmap.png'), dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
failure_labels = ['correct', 'planning', 'retrieval_query', 'retrieval_context', 'reasoning_or_synthesis', 'tool_or_format', 'no_finish']
x = np.arange(len(failure_labels))
width = 0.8 / max(1, len(df_failure))
for i, (_, row) in enumerate(df_failure.iterrows()):
    vals = [row[label] for label in failure_labels]
    ax.bar(x + i * width, vals, width, label=f"{row['model']}/{row['dataset']}")
ax.set_xticks(x + width * max(0, len(df_failure) - 1) / 2)
ax.set_xticklabels(failure_labels, rotation=20, ha='right')
ax.set_ylabel('Trajectories')
ax.set_title('Phase 11_b failure-family breakdown')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR_B, 'phase11b_failure_breakdown.png'), dpi=120)
plt.show()
